In [ ]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd

Mounted at /content/drive


### Mengatasi Error Otorisasi Hugging Face

Jika Anda menemui error `401 Unauthorized` saat memuat model dari Hugging Face Hub (misalnya `indobenchmark/indobertweet-sentiment`), itu berarti Anda perlu melakukan otentikasi. Anda bisa menyediakan token Hugging Face Anda dengan dua cara:

1.  **Menggunakan `huggingface_hub.login()`:** Ini akan meminta Anda untuk memasukkan token Anda secara interaktif.
2.  **Menyimpan token sebagai Colab Secret:** Cara yang lebih aman untuk penggunaan jangka panjang. Beri nama secret Anda `HF_TOKEN`.

Pilih salah satu cara di bawah ini:


In [ ]:
from huggingface_hub import login, HfFolder

# --- Pilihan 1: Login interaktif (akan meminta token Anda di output) ---
# login()

# --- Pilihan 2: Menggunakan Colab Secret ---
# Pastikan Anda sudah menyimpan token Hugging Face Anda dengan nama 'HF_TOKEN' di Colab Secrets.
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        HfFolder.save_token(hf_token)
        print("Hugging Face token loaded from Colab secrets.")
    else:
        print("Hugging Face token not found in Colab secrets. Please add it or use login().")
except Exception as e:
    print(f"Could not load HF_TOKEN from Colab secrets: {e}")
    print("Please ensure 'HF_TOKEN' is set in Colab Secrets or use huggingface_hub.login().")

Setelah token disiapkan, kita dapat memuat ulang model yang di-fine-tune untuk sentimen.

In [ ]:
# 1. Load Model dan Tokenizer
# Model yang sudah di fine tuning untuk sentimen.
model_name = "indobenchmark/indobertweet-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# 2. Inisialisasi Pipeline Sentiment Analysis
# Kita menggunakan pipeline agar proses inferensi lebih sederhana
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer
)

# 3. Contoh Data Hasil Preprocessing
# Memuat data dari CSV
try:
    df = pd.read_csv('/content/drive/MyDrive/Kuliah/Semester 6/Jurnal/komentar_youtube_clean.csv')
    data_clean = df['clean_text'].dropna().tolist()
    print(f"Data berhasil diakses. Total {len(data_clean)} komentar bersih.")
except FileNotFoundError:
    print("Error: file CSV tidak ditemukan.")
    data_clean = [] # Set to empty list or handle error as appropriate
except KeyError:
    print("Error: 'clean_text' kolom tidak ditemukan di file CSV tersebut.")
    data_clean = []
except Exception as e:
    print(f"Error saat membaca file CSV: {e}")
    data_clean = []


# 4. Fungsi untuk Auto-Labeling
def auto_label(texts):
    results = sentiment_classifier(texts)
    labels = []

    for res in results:
        # Mapping label dari model ke teks yang mudah dibaca
        # Model ini biasanya outputnya LABEL_0 (Negatif), LABEL_1 (Netral), LABEL_2 (Positif)
        label_map = {
            'LABEL_0': 'Negatif',
            'LABEL_1': 'Netral',
            'LABEL_2': 'Positif'
        }
        labels.append(label_map.get(res['label'], res['label']))

    return labels

In [6]:
# 1. Load Model dan Tokenizer
# Lebih khusus ke sentiment yang sudah di fine tuning 'indobenchmark/indobertweet-sentiment'
model_name = "indolem/indobertweet-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# 2. Inisialisasi Pipeline Sentiment Analysis
# Kita menggunakan pipeline agar proses inferensi lebih sederhana
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer
)

# 3. Contoh Data Hasil Preprocessing
# Memuat data dari CSV
try:
    df = pd.read_csv('/content/drive/MyDrive/Kuliah/Semester 6/Jurnal/komentar_youtube_clean.csv')
    data_clean = df['clean_text'].dropna().tolist()
    print(f"Data berhasil diakses. Total {len(data_clean)} komentar bersih.")
except FileNotFoundError:
    print("Error: file CSV tidak ditemukan.")
    data_clean = [] # Set to empty list or handle error as appropriate
except KeyError:
    print("Error: 'clean_text' kolom tidak ditemukan di file CSV tersebut.")
    data_clean = []
except Exception as e:
    print(f"Error saat membaca file CSV: {e}")
    data_clean = []


# 4. Fungsi untuk Auto-Labeling
def auto_label(texts):
    results = sentiment_classifier(texts)
    labels = []

    for res in results:
        # Mapping label dari model ke teks yang mudah dibaca
        # Model ini biasanya outputnya LABEL_0 (Negatif), LABEL_1 (Netral), LABEL_2 (Positif)
        label_map = {
            'LABEL_0': 'Negatif',
            'LABEL_1': 'Netral',
            'LABEL_2': 'Positif'
        }
        labels.append(label_map.get(res['label'], res['label']))

    return labels

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Data berhasil diakses. Total 10990 komentar bersih.


In [7]:
# 5. Eksekusi
labels = auto_label(data_clean)

# Tampilkan Hasil
for text, label in zip(data_clean, labels):
    print(f"Teks: {text} | Label: {label}")

Teks: penipu betul perempuan blonde ini sama dengan trumpet | Label: Negatif
Teks: tidak boleh diterima????? tanpa rasa malu penyataan itu keluar | Label: Negatif
Teks: konoha konoha beritanya berbohong terus | Label: Negatif
Teks: pbb cuma bisa mengecam saja tapi tidak bisa bertindak tololl korban sudah tidak bisa terhitung tangkep tuh dalangnya sih netanyahu | Label: Negatif
Teks: amerika rasis | Label: Negatif
Teks: ini politik konoha tidak bisa selami apa 7 an sesungguhx dari usah | Label: Negatif
Teks: palestin memerlukan 100thn untuk dibina semula atau lebih lama lagi | Label: Negatif
Teks: ya allah lindungi negara iran berikanlah kemenagan buatnya islam pasti berjaya akhirnya | Label: Negatif
Teks: musibat punya trump | Label: Netral
Teks: sih israel tuh jinakan dulu biar cepat damai | Label: Netral
Teks: aneh ini media penggiringan opini sejak kapan negara sekelas as memohon modal foto video tanpa suara di beri opini memohon | Label: Negatif
Teks: tutup saja selat hormuz hanya 

In [8]:
label_distribution = pd.Series(labels).value_counts()
print("Persebaran Label:")
print(label_distribution)

Persebaran Label:
Negatif    9600
Netral     1390
Name: count, dtype: int64
